# Rag Pipeline
- Data ingestion to vector DB pipeline

In [5]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Admin\AppData\Local\Temp\ipykernel_6744\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


## Read all the PDF's

In [6]:
def preprocess_all_pdf(pdf_dir):
    all_docs=[]
    pdf_dir=Path(pdf_dir)

    # find all files
    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader=PyMuPDFLoader(str(pdf_file))
            docs=loader.load()

            # Add source info to metdata
            for doc in docs:
                doc.metadata['source']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_docs.extend(docs)
            print(f"Loaded {len(docs)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    
    print(f"Total pages loaded: {len(all_docs)}")
    return all_docs

In [33]:
all_pdf_docs=preprocess_all_pdf('../data')

Found 5 PDF files to process
Processing Machine_Learning_Unit_1.pdf
Loaded 29 pages
Processing Machine_Learning_Unit_2.pdf
Loaded 46 pages
Processing Machine_Learning_Unit_3.pdf
Loaded 28 pages
Processing Machine_Learning_Unit_4.pdf
Loaded 32 pages
Processing Machine_Learning_Unit_5.pdf
Loaded 29 pages
Total pages loaded: 164


In [8]:
all_pdf_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'TeX', 'creationdate': '2025-09-21T10:02:46+00:00', 'source': 'Machine_Learning_Unit_2.pdf', 'file_path': '..\\data\\Machine_Learning_Unit_2.pdf', 'total_pages': 46, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-14T21:03:01+05:30', 'trapped': '', 'modDate': "D:20251214210301+05'30'", 'creationDate': 'D:20250921100246Z', 'page': 0, 'file_type': 'pdf'}, page_content='UNIT 2 Linear and Logistic Regression,\nBayesian Learning, Support Vector Machines\nMLT, BCS 055\n1')

## Chunking: Text splitting

In [9]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks\n")

    if split_docs:
        print("Example of chunks:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}\n")
    
    return split_docs

In [34]:
chunks=split_documents(all_pdf_docs)

Split 164 documents into 267 chunks

Example of chunks:
Content: Introduction to Machine Learning
And Its Types
Introduction to Machine Learning- Unit 1
Machine Learning (ML) is a subfield of Artificial Intelligence (AI) that enables systems to
learn from data, imp...
Metadata: {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-08-31T09:26:43+00:00', 'source': 'Machine_Learning_Unit_1.pdf', 'file_path': '..\\data\\Machine_Learning_Unit_1.pdf', 'total_pages': 29, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-14T19:25:53+05:30', 'trapped': '', 'modDate': "D:20251214192553+05'30'", 'creationDate': 'D:20250831092643Z', 'page': 0, 'file_type': 'pdf'}



## Embedding and VectorDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# resposible for embedding
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        '''
        initialize the embedding manager
        Args:
            model_name (str): Huggingface transformer model for sentence embeddings.
        '''
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def gen_embeddings(self, texts: List[str]) -> np.ndarray:
        '''
        Generate embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to embed.
        Returns:
            np.ndarray: Array of embeddings, with shape of (len(texts), embedding_dimension).
        '''
        if not self.model:
            raise ValueError("Model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [35]:
# initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3370.68it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [14]:
# managing documnet embeddings in chromadb vector store
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        '''
        Initialize the vector store.
        Args:
            collection_name (str): Name of the collection in ChromaDB.
            persist_directory (str): Directory to persist the ChromaDB data.
        '''
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        #  Initialize ChromaDB client and collection
        try:
            # create persitent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            # print(f"Existing documents in the collection: {len(self.collection.get()['ids'])}")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''
        Add documents and their embeddings to the vector store.
        Args:
            documents (List[Any]): List of document objects (e.g., LangChain Document).
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        '''
        if not self.collection:
            raise ValueError("Collection is not initialized.")

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store.")

        # Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        docs_text=[]
        embeddings_list=[]

        # Add to collection
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # generate unique id for each document
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            # doc content
            docs_text.append(doc.page_content)
            # embeddings
            embeddings_list.append(emb.tolist())
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=docs_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e

In [36]:
vector_store=VectorStore()

Vector store initialized with collection: pdf_documents
Existing documents in the collection: 196


### Converting text to embedding

In [37]:
texts=[doc.page_content for doc in chunks]

# Generate embeddings 
embeddings=embedding_manager.gen_embeddings(texts)

# store embeddings in vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 267 texts.


Batches: 100%|██████████| 9/9 [00:39<00:00,  4.38s/it]


Generated embeddings with shape: (267, 384)
Adding 267 documents to the vector store.
Successfully added 267 documents to the vector store.
Total documents in the collection after addition: 463


## Retriver pipeline from VectorStore

In [17]:
class RAGretriver:
    # Handles query based retrieval from the vector store
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        '''
        Initialize the retriver
        Args:
            vector_store (VectorStore): Instance of the VectorStore class/containing doc embeddings.
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager class.
        '''
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        '''
        Retrieve top_k relevant documents for the given query.
        Args:
            query (str): The query string.
            top_k (int): Number of top documents to retrieve.
            score_threshold (float): Minimum similarity score for retrieved documents.
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata.
        '''
        print(f"Retrieving for query: {query}")
        print(f"Top_k: {top_k}, Score threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding=self.embedding_manager.gen_embeddings([query])[0]

        # search in the vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
                # include=['metadatas', 'documents', 'distances']
            )

            # Process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                docs=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                # print(f"Raw results - Distances: {distances}")  # Debug: show all distances

                for i, (doc, metadata, distance, doc_id) in enumerate(zip(docs, metadatas, distances, ids)):
                    sim_score=1 - distance  # Convert distance to similarity score

                    # print(f"Doc {i+1}: similarity={sim_score:.4f}, threshold={score_threshold}")  # Debug

                    if sim_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': doc,
                            'metadata': metadata,
                            'sim_score': sim_score,
                            'distance': distance,
                            'Rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents after filtering.")
            else:
                print("No documents found in the vector store for the given query.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise e

In [18]:
rag_retriever=RAGretriver(vector_store, embedding_manager)
rag_retriever

In [40]:
results = rag_retriever.retrieve("Decision Tree")

Retrieving for query: Decision Tree
Top_k: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.28it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents after filtering.


## Integration VectorDB context pipeline with LLM output

In [29]:
#  Simple RAG pipeline with nemotron 3
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize OpenAI client
nemotron_api_key = os.getenv("NVIDIA_API_KEY")

llm=ChatOpenAI(
    api_key=nemotron_api_key,
    base_url=os.getenv("NVIDIA_BASE_URL"),
    model=os.getenv("MODEL_NAME"),
    temperature=0.1,
    max_tokens=1024
)

In [30]:
#  Simple rag function
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve relevant documents
    results=retriever.retrieve(query, top_k=top_k)

    if not results:
        print("No relevant documents found for the query.")
        return None

    # Prepare context for the LLM
    context="\n\n".join([doc['content'] for doc in results])

    # generate answer from the LLM
    prompt=f"""Answer the following question concisely based on the provided context:
            Context:
            {context}
            Question: {query}
            Answer:
            """

    # Generate response from LLM
    response=llm.invoke(prompt)
    return response.content

In [41]:
answer=rag_simple("what is a Regression in machine learning?", rag_retriever, llm, top_k=3)
print(f"Answer:\n{answer}")

Retrieving for query: what is a Regression in machine learning?
Top_k: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after filtering.


Answer:
Regression is a supervised machine learning method used to predict a target value based on input data. It includes **Linear Regression** for predicting continuous numbers (e.g., price, salary) and **Logistic Regression** for predicting categories (e.g., Yes/No, spam/not spam).


## Enhance RAG pipeline

In [42]:
def rag_advance(query, retriever, llm, top_k=5, return_context=False):
    '''
    RAG pipleine with extra features
        - returns: answer, sources, confidence score, context (optional)
    '''
    # Retrieve relevant documents
    results=retriever.retrieve(query, top_k=top_k)

    if not results:
        print("No relevant documents found for the query.")
        return None

    # Prepare context for the LLM
    context="\n\n".join([doc['content'] for doc in results])
    sources=[{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page': doc['metadata'].get('page', 'Unknown'),
        'score': doc['sim_score'],
        'preview': doc['content'][:200] + '...' if len(doc['content']) > 200 else doc['content']
    } for doc in results]
    confidence=max([doc['sim_score'] for doc in results])

    # generate answer from the LLM
    prompt=f"""Answer the following question concisely based on the provided context:
            Context:
            {context}
            Question: {query}
            Answer:
            """

    # Generate response from LLM
    response=llm.invoke(prompt)
    output={
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    return output

In [53]:
response=rag_advance("How CNN can helpful for Diabetic Retinopathy?", rag_retriever, llm, top_k=5, return_context=True)
if response:
    print(f"Answer:\n{response['answer']}\n")
    print(f"Confidence Score: {response['confidence']:.4f}\n")
    print(f"Sources: {response['sources']}\n")
    print(f"Context:\n{response['context'][:300]}...\n")  # Print first 300 characters of context

Retrieving for query: How CNN can helpful for Diabetic Retinopathy?
Top_k: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.28it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents after filtering.
Answer:
CNNs automatically analyze high-resolution retinal fundus images to classify DR severity levels (No DR, Mild, Moderate, Severe, Proliferative) with high accuracy. They provide a reliable, automated screening method that reduces the time consumption and variability associated with manual ophthalmologist screening, enabling early detection to prevent vision loss.

Confidence Score: 0.6265

Sources: [{'source': 'Machine_Learning_Unit_4.pdf', 'page': 24, 'score': 0.6265246570110321, 'preview': 'CONTENTS\n24\n0.11.5\nSummary\n• Training adjusts weights to minimize error.\n• Forward propagation computes outputs, backward propagation updates weights.\n• Proper choice of hyperparameters and regulariza...'}, {'source': 'Machine_Learning_Unit_4.pdf', 'page': 24, 'score': 0.36744463443756104, 'preview': '• Input: High-resolution retinal fundus images (color images of the retina).\n• Output: Classification into one of the DR severity levels:\n1. No DR

In [54]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
import time
class AdvanceRAGPipeline:
    def __init__(self, retriever: RAGretriver, llm: ChatOpenAI):
        self.retriever=retriever
        self.llm=llm
        self.history=[] # To store query history

    def query(self, query: str, top_k: int=5, min_score: float=0.1, stream: bool=False, summarize: bool=False)-> Dict[str, Any]:
        # retrieve relevant docs
        results=self.retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

        if not results:
            print("No relevant documents found for the query.")
            return None

        context='\n\n'.join([doc['content'] for doc in results])
        sources=[{
            'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['sim_score'],
            'preview':doc['content'][:200] + '...' if len(doc['content']) > 200 else doc['content']
        } for doc in results]
        prompt=f"""Answer the following question concisely based on the provided context:
            Context:
            {context}
            Question: {query}
            Answer:
            """
        if stream:
            print("Streaming answer: ")
            for i in range(0, len(prompt), 80):
                print(prompt[i:i+80], end='', flush=True)
                time.sleep(0.05)
            print()

        response=self.llm.invoke(prompt)
        answer=response.content

        # add citation to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'query': query,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'query': query,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [57]:
adv_rag = AdvanceRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Decision Tree", top_k=3, min_score=0.1, stream=True, summarize=True)
if result:
    print("\nFinal Answer:", result['answer'])
    print("Summary:", result['summary'])
    print("History:", result['history'][-1])

Retrieving for query: what is Decision Tree
Top_k: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after filtering.
Streaming answer: 
Answer the following question concisely based on the provided context:
            Context:
            9. Repeat until all instances are classified or no attributes are left.
10. Return the constructed Decision Tree.
ISSUES IN DECISION TRE

E LEARNING
Decision Trees are widely used for classification and regression tasks due to their inter-
pretability and simplicity. However, despite their advantages, Decision Trees suffer from
several issues that can affect their performance, reliability, and generalization. Understand-
ing these issues is crucial for effectively applying Decision Trees in practical scenarios.
1. Overfitting
• Occurs when the Decision Tree model learns noise or minor variations in the train-
ing dataset rather than the underlying pattern.
• Leads to high accuracy on training data but poor performance on unseen data.
5

9. Repeat until all instances are classified or no attributes are left.
10. Return the constructed Decision Tree.
ISSUES IN DECISION TREE LEARNING
Decision Trees are widely used for classification and regression tasks due to their inter-
pretability and simplicity. However, despite their advantages, Decision Trees suffer from
several issues that can affect their performance, reliability, 